In [18]:
from pathlib import Path

import pandas as pd
import numpy as np

DATA_DIR = Path("/Users/anmoulmalhotra/Documents/ProjectDissertationDir/hybrid-av/data/drebin/")

df = pd.read_csv(DATA_DIR / "drebin215dataset5560malware9476benign.csv")

df.head()

/var/folders/zf/7vrgv7dn7qsbb8w1fnmwts7w0000gn/T/ipykernel_8207/3788122002.py:8: DtypeWarning: Columns (0: TelephonyManager.getSimCountryIso ) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA_DIR / "drebin215dataset5560malware9476benign.csv")


,transact,onServiceConnected,bindService,attachInterface,ServiceConnection,android.os.Binder,SEND_SMS,Ljava.lang.Class.getCanonicalName,Ljava.lang.Class.getMethods,Ljava.lang.Class.cast,...,READ_CONTACTS,DEVICE_POWER,HARDWARE_TEST,ACCESS_WIFI_STATE,WRITE_EXTERNAL_STORAGE,ACCESS_FINE_LOCATION,SET_WALLPAPER_HINTS,SET_PREFERRED_APPLICATIONS,WRITE_SECURE_SETTINGS,class
0,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,1,0,0,0,0,S
1,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,1,0,0,0,0,S
2,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,S
3,0,0,0,0,0,0,0,0,0,1,...,0,0,0,1,1,1,0,0,0,S
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,1,0,0,0,S


In [12]:
#sanity checks
df.shape, df.columns[:10]
df['class'].value_counts()
df.isna().sum().head()

transact               0
onServiceConnected     0
bindService            0
attachInterface        0
ServiceConnection      0
dtype: int64

In [13]:
#Clean up mixed / non-numeric values in Drebin features

# 1 Replace '?' with 0
df = df.replace('?', 0)

# 2 Make sure all feature columns except 'class' are numeric
feature_cols = [c for c in df.columns if c != 'class']

for c in feature_cols:
    # change to numeric, turn any leftover non numeric into 0, then cast to int
    df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0).astype(int)

#quick sanity check
print(df.dtypes.head(10))
print()

transact                              int64
onServiceConnected                    int64
bindService                           int64
attachInterface                       int64
ServiceConnection                     int64
android.os.Binder                     int64
SEND_SMS                              int64
Ljava.lang.Class.getCanonicalName     int64
Ljava.lang.Class.getMethods           int64
Ljava.lang.Class.cast                 int64
dtype: object



In [14]:
from sklearn.model_selection import train_test_split

# Separate features and label
X = df.drop(columns=['class'])
y = df['class']

# 60% train, 20% val, 20% test (stratified)
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.4,
    stratify=y,
    random_state=42,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=42,
)

len(y_train), len(y_val), len(y_test), y_train.value_counts(normalize=True)

(9021,
 3007,
 3008,
 class
 B    0.630196
 S    0.369804
 Name: proportion, dtype: float64)

In [15]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    n_jobs=-1,
    random_state=42,
)

rf.fit(X_train, y_train)

y_val_pred = rf.predict(X_val)

val_acc = accuracy_score(y_val, y_val_pred)
val_f1_macro = f1_score(y_val, y_val_pred, average='macro')

print(f"Validation accuracy: {val_acc:.4f}")
print(f"Validation Macro-F1: {val_f1_macro:.4f}")
print()
print("Validation classification report:")
print(classification_report(y_val, y_val_pred))

Validation accuracy: 0.9757
Validation Macro-F1: 0.9738

Validation classification report:
              precision    recall  f1-score   support

           B       0.97      0.99      0.98      1895
           S       0.99      0.95      0.97      1112

    accuracy                           0.98      3007
   macro avg       0.98      0.97      0.97      3007
weighted avg       0.98      0.98      0.98      3007



In [16]:
y_test_pred = rf.predict(X_test)

test_acc = accuracy_score(y_test, y_test_pred)
test_f1_macro = f1_score(y_test, y_test_pred, average='macro')

print(f"Test accuracy: {test_acc:.4f}")
print(f"Test Macro-F1: {test_f1_macro:.4f}")

Test accuracy: 0.9767
Test Macro-F1: 0.9748


In [22]:
from joblib import dump

MODEL_DIR = Path("/Users/anmoulmalhotra/Documents/ProjectDissertationDir/hybrid-av/saved/models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

model_path = MODEL_DIR / "drebin_static_rf.pkl"
dump(rf, model_path)

model_path

PosixPath('/Users/anmoulmalhotra/Documents/ProjectDissertationDir/hybrid-av/saved/models/drebin_static_rf.pkl')